# 03 — CLIP@384 + EMA + diagnostics

Upgrade of `02_train_evaluate_visualize_colab_unfrozen_5.ipynb`. Direct A/B with the 64.27% baseline; same vocab (3129), same splits, same soft-target/eval metric.

**Deltas vs baseline**
1. Vision: `torchvision` ViT-B/16 @ 224 (ImageNet) → CLIP ViT-B/16 @ **384** via `open_clip` + LAION-2B (`laion2b_s34b_b88k`), CLIP normalization, position embeddings interpolated to 24×24+1=577 tokens.
2. Image cache: new `vqa_images_384.h5` (384×384 uint8) built from raw JPEGs — no 256→384 upsampling.
3. Text encoder: explicit `last_hidden_state` (pooler bypassed) + diagnostic print.
4. EMA of all weights (decay 0.9998); eval on EMA, save both `*_live.pt` and `*_ema.pt`.
5. VQA-safe light augmentation: RandAugment(n=2, m=9, **no geometric ops**) + ColorJitter(0.1) + RandomErasing(p=0.25); clean val.
6. Per-question-type val each epoch using `ann["answer_type"]` (yes/no | number | other) from VQA-v2 annotations.
7. Modern `torch.amp` API; gradient-checkpointing flag for the CLIP vision tower; grad accumulation.
8. 12 epochs, warmup = 5% of total optimizer steps, effective batch 64 (per-step 16 × accum 4); OOM fallback documented.

Expected outcome: **67–69% val VQA acc** at convergence, with per-type breakdown showing where the remaining gap sits.


## Load and Unzip VQA Dataset


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!rm -rf /content/sample_data/


In [ ]:
# Get vqa data from drive opt.
!cp "/content/drive/MyDrive/VQA.zip" "/content/"


In [ ]:
# make data dirs
!mkdir -p /content/data/
!mkdir -p /content/data/answers
!mkdir -p /content/data/images
!mkdir -p /content/data/questions

#copy over zips (https://drive.google.com/drive/folders/1VJ1xNxo_dAGJ4ZcpaFQBo-wpdIChZkpx?usp=sharing) from drive into here
!cp -r /content/drive/MyDrive/VQA/ /content/data/zip/


In [ ]:
# unzip vqa.zip, then remove it opt.
!unzip -q /content/VQA.zip -d /content/data/zip

!rm -rf /content/VQA.zip


In [ ]:
# test2015 opt. (not used at train time; kept for parity with baseline)
!unzip -q /content/data/zip/test2015.zip -d /content/data/images/

!rm -rf /content/data/zip/test2015.zip


In [ ]:
# train2014
!unzip -q /content/data/zip/train2014.zip -d /content/data/images/

!rm -rf /content/data/zip/train2014.zip


In [ ]:
# val2014
!unzip -q /content/data/zip/val2014.zip -d /content/data/images/

!rm -rf /content/data/zip/val2014.zip


In [ ]:
# extract annotations (labels)
!unzip -q /content/data/zip/v2_Annotations_Train_mscoco.zip -d /content/data/answers/
!unzip -q /content/data/zip/v2_Annotations_Val_mscoco.zip -d /content/data/answers/

!rm -rf /content/data/zip/v2_Annotations_Train_mscoco.zip
!rm -rf /content/data/zip/v2_Annotations_Val_mscoco.zip


In [ ]:
# extract questions
!unzip -q /content/data/zip/v2_Questions_Test_mscoco.zip -d /content/data/questions/
!unzip -q /content/data/zip/v2_Questions_Train_mscoco.zip -d /content/data/questions/
!unzip -q /content/data/zip/v2_Questions_Val_mscoco.zip -d /content/data/questions/

!rm -rf /content/data/zip/v2_Questions_Test_mscoco.zip
!rm -rf /content/data/zip/v2_Questions_Train_mscoco.zip
!rm -rf /content/data/zip/v2_Questions_Val_mscoco.zip


In [ ]:
# (Optional) copy any past results for the resume path.
!mkdir -p /content/results
# !cp -r /content/drive/MyDrive/clip_upgrade_results/* /content/results/  # uncomment when you have prior CLIP-upgrade runs


# 2 — Train, Evaluate & Visualize (CLIP@384)

Pip installs and imports next. `open_clip_torch` is the only addition over the baseline.


In [ ]:
!pip install -q torch torchvision transformers open_clip_torch matplotlib tqdm Pillow h5py


In [ ]:
import json
import math
import random
import shutil
import time
from collections import Counter, defaultdict
from contextlib import nullcontext
from pathlib import Path

import h5py
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from PIL import Image
import open_clip
# Modern AMP API (replaces deprecated torch.cuda.amp)
from torch.amp import GradScaler, autocast
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.transforms import RandAugment
from tqdm import tqdm
from transformers import RobertaModel, RobertaTokenizer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")


## Configuration


In [ ]:
# Paths
DATA_DIR       = Path("/content/data")
CHECKPOINT_DIR = Path("/content/results/checkpoints")
METRICS_DIR    = Path("/content/results/metrics")
FIGURES_DIR    = Path("/content/results/figures")

# Model
NUM_ANSWERS      = 3129    # standard VQAv2 vocab, ~93% coverage (kept identical to baseline)
EMBED_DIM        = 512
NUM_HEADS        = 8
DROPOUT          = 0.3     # FFN/projection dropout
ATTN_DROPOUT     = 0.2     # cross-attention softmax dropout
CLS_DROPOUT      = 0.5     # final classifier head dropout

# Vision encoder (CLIP)
IMAGE_SIZE       = 384
CLIP_MODEL       = "ViT-B-16"
CLIP_PRETRAINED  = "laion2b_s34b_b88k"
USE_GRAD_CHECKPOINT = False   # flip True if you OOM at BATCH_SIZE=16

# Data
MAX_QUESTION_LEN = 20
MAX_SAMPLES      = None    # set to a few thousand for a dev run

# Training
BATCH_SIZE        = 16     # per-step batch
GRAD_ACCUM_STEPS  = 4      # effective batch = BATCH_SIZE * GRAD_ACCUM_STEPS = 64
LEARNING_RATE     = 1e-4   # fusion + classifier LR
ENCODER_LR        = 1e-5   # CLIP vision + RoBERTa LR (10x smaller)
WARMUP_FRAC       = 0.05   # 5% of total optimizer steps (was 1 epoch / ~5.5% in baseline)
WEIGHT_DECAY      = 1e-2
EPOCHS            = 12     # was 18; we expect to converge faster with better features
NUM_WORKERS       = 4
SEED              = 42
USE_AMP           = True

# EMA
USE_EMA           = True
EMA_DECAY         = 0.9998

# Gradient clipping
GRAD_CLIP_NORM    = 1.0    # mild, safety against any AMP loss spike

for d in [CHECKPOINT_DIR, METRICS_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(SEED)
print(f"Effective batch = {BATCH_SIZE * GRAD_ACCUM_STEPS} | epochs = {EPOCHS} | image size = {IMAGE_SIZE}")


## Answer vocab, normalization, transforms

Vocab and `normalize_answer` are **byte-identical** to the baseline (cells 20+24) so train/val splits and the answer-id space match.

`RandAugmentNoGeom` is a tiny subclass of torchvision's `RandAugment` that drops geometric ops (Translate, Shear, Rotate). VQA answers depend on spatial position (e.g. "what is on the left of..."), so these would silently corrupt the supervision signal.


In [ ]:
import re

# VQAv2 official-style answer normalization (lowercase, strip punctuation,
# remove articles, fix common contractions). Applied identically when building
# the vocabulary and when constructing soft targets so they stay in sync.
# (Identical to baseline cell 20.)
_ARTICLES = {"a", "an", "the"}
_PUNCT_RE = re.compile(r"[^\w\s\']")
_CONTRACTIONS = {
    "dont": "don't", "doesnt": "doesn't", "didnt": "didn't",
    "isnt": "isn't", "arent": "aren't",
    "wasnt": "wasn't", "werent": "weren't",
    "wont": "won't", "cant": "can't", "couldnt": "couldn't",
    "wouldnt": "wouldn't", "shouldnt": "shouldn't",
    "havent": "haven't", "hasnt": "hasn't", "hadnt": "hadn't",
    "thats": "that's", "whats": "what's", "wheres": "where's",
    "theres": "there's", "heres": "here's", "youre": "you're",
    "theyre": "they're", "weve": "we've", "youve": "you've",
}


def normalize_answer(s: str) -> str:
    s = s.lower().strip()
    s = _PUNCT_RE.sub(" ", s)
    s = " ".join(t for t in s.split() if t not in _ARTICLES)
    return _CONTRACTIONS.get(s, s)


def build_answer_vocab(annotations_file, top_k=3129):
    """Build answer vocabulary from the top_k most frequent normalized answers.

    Counts every annotator answer (10 per question), not just the
    `multiple_choice_answer`, which gives a more representative distribution.
    """
    with open(annotations_file) as f:
        annotations = json.load(f)["annotations"]

    counter = Counter()
    for ann in annotations:
        for a in ann["answers"]:
            counter[normalize_answer(a["answer"])] += 1

    most_common = [ans for ans, _ in counter.most_common(top_k)]
    answer_to_idx = {ans: idx for idx, ans in enumerate(most_common)}
    idx_to_answer = {idx: ans for ans, idx in answer_to_idx.items()}
    return answer_to_idx, idx_to_answer


# VQA-v2 official answer-type taxonomy: "yes/no", "number", "other".
# Pulled from each annotation's `answer_type` field; no need to parse questions.
ANSWER_TYPES = ("yes/no", "number", "other")
ANSWER_TYPE_TO_IDX = {t: i for i, t in enumerate(ANSWER_TYPES)}


CLIP_MEAN = (0.48145466, 0.4578275, 0.40821073)
CLIP_STD  = (0.26862954, 0.26130258, 0.27577711)


class RandAugmentNoGeom(RandAugment):
    """RandAugment without geometric ops (Translate/Shear/Rotate).

    VQA labels frequently depend on spatial position ("what's on the left of...",
    "how many people are standing"). Geometric augmentations can move answer-
    relevant content out of frame or silently change spatial relations.
    """

    _DROP_OPS = {"TranslateX", "TranslateY", "ShearX", "ShearY", "Rotate"}

    def _augmentation_space(self, num_bins, image_size):
        space = super()._augmentation_space(num_bins, image_size)
        return {k: v for k, v in space.items() if k not in self._DROP_OPS}


def get_train_transform():
    """384x384 in, CLIP-normalized out. H5 already stores 384x384 -> no resize."""
    return transforms.Compose([
        transforms.RandomHorizontalFlip(p=0.5),
        RandAugmentNoGeom(num_ops=2, magnitude=9,
                          interpolation=transforms.InterpolationMode.BICUBIC),
        transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.0),
        transforms.ToTensor(),
        transforms.Normalize(mean=CLIP_MEAN, std=CLIP_STD),
        transforms.RandomErasing(p=0.25, scale=(0.02, 0.10), ratio=(0.3, 3.3), value=0),
    ])


def get_val_transform():
    return transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=CLIP_MEAN, std=CLIP_STD),
    ])


def get_h5_preprocess_transform(size=IMAGE_SIZE):
    """Resize+CenterCrop to size x size for HDF5 storage. Bicubic to preserve
    detail when going from typical COCO resolutions down/up to 384."""
    return transforms.Compose([
        transforms.Resize(size, interpolation=transforms.InterpolationMode.BICUBIC),
        transforms.CenterCrop(size),
    ])


## HDF5 image cache at 384×384

We use a **new** cache `vqa_images_384.h5` so the original 256-px cache stays intact for the baseline notebook. ~50 GB on local disk (384²×3 bytes × ~123k images). The build step prints expected size and elapsed time. If you already have this file on Drive, drop it at `/content/drive/MyDrive/VQA_cache/vqa_images_384.h5` and this cell will fast-path the copy.


In [ ]:
# --- HDF5 Preprocessing at 384x384 (built from raw JPEGs, NO 256->384 upsampling) ---
DRIVE_H5_384 = Path("/content/drive/MyDrive/VQA_cache/vqa_images_384.h5")
LOCAL_H5_384 = DATA_DIR / "vqa_images_384.h5"


def _build_h5_at_size(out_path: Path, size: int):
    preprocess = get_h5_preprocess_transform(size=size)
    image_dirs = [DATA_DIR / "images" / "train2014", DATA_DIR / "images" / "val2014"]

    all_paths = []
    for d in image_dirs:
        if d.exists():
            all_paths.extend(sorted(d.glob("*.jpg")))
    n = len(all_paths)
    est_gb = (n * size * size * 3) / 1e9
    print(f"Found {n:,} images. Building H5 @ {size}x{size} -> ~{est_gb:.1f} GB on disk.")

    with h5py.File(out_path, "w") as h5f:
        imgs_ds = h5f.create_dataset(
            "images", shape=(n, size, size, 3),
            dtype=np.uint8, chunks=(1, size, size, 3),
        )
        ids_ds = h5f.create_dataset("image_ids", shape=(n,), dtype=np.int64)

        for i, path in enumerate(tqdm(all_paths, desc=f"Building H5@{size}")):
            image_id = int(path.stem.split("_")[-1])
            img = Image.open(path).convert("RGB")
            img = preprocess(img)
            imgs_ds[i] = np.array(img)
            ids_ds[i] = image_id

    print(f"Saved {n:,} images to {out_path}")


if LOCAL_H5_384.exists():
    print(f"[H5] using existing local cache: {LOCAL_H5_384}")
elif DRIVE_H5_384.exists():
    print(f"[H5] copying from Drive: {DRIVE_H5_384} -> {LOCAL_H5_384}")
    shutil.copy2(DRIVE_H5_384, LOCAL_H5_384)
else:
    t0 = time.time()
    _build_h5_at_size(LOCAL_H5_384, IMAGE_SIZE)
    print(f"[H5] build elapsed: {time.time() - t0:.0f}s")
    # Optional: persist to Drive for future runs
    # DRIVE_H5_384.parent.mkdir(parents=True, exist_ok=True)
    # shutil.copy2(LOCAL_H5_384, DRIVE_H5_384)


## VQADataset

H5-at-384 reader. Returns one extra field — `answer_type_idx` — so per-type val can stream alongside the main eval without re-parsing questions or re-loading annotations.


In [ ]:
class VQADataset(Dataset):
    """VQA v2 dataset with H5 image loading and soft targets.

    Returns (image_tensor, input_ids, attention_mask, answer_target, answer_type_idx).
    answer_type_idx is an int in [0, len(ANSWER_TYPES)) corresponding to
    "yes/no", "number", "other" — VQA-v2's official taxonomy.
    """

    def __init__(self, questions_file, annotations_file, h5_path,
                 answer_to_idx=None, top_k_answers=3129,
                 max_question_len=20, transform=None, max_samples=None):
        self.h5_path = Path(h5_path)
        self.max_question_len = max_question_len
        self.transform = transform or get_val_transform()
        self.tokenizer = RobertaTokenizer.from_pretrained("roberta-base")
        self._h5f = None  # lazy-opened per DataLoader worker

        if answer_to_idx is None:
            self.answer_to_idx, self.idx_to_answer = build_answer_vocab(
                annotations_file, top_k_answers)
        else:
            self.answer_to_idx = answer_to_idx
            self.idx_to_answer = {v: k for k, v in answer_to_idx.items()}

        with h5py.File(self.h5_path, "r") as f:
            image_ids = f["image_ids"][:]
        self.id_to_row = {int(iid): i for i, iid in enumerate(image_ids)}

        with open(questions_file) as f:
            questions_data = json.load(f)["questions"]
        with open(annotations_file) as f:
            annotations_data = json.load(f)["annotations"]

        ann_by_qid = {ann["question_id"]: ann for ann in annotations_data}
        num_answers = len(self.answer_to_idx)

        self.samples = []
        for q in questions_data:
            ann = ann_by_qid.get(q["question_id"])
            if ann is None:
                continue

            target = torch.zeros(num_answers, dtype=torch.float)
            for a in ann["answers"]:
                ans_text = normalize_answer(a["answer"])
                idx = self.answer_to_idx.get(ans_text)
                if idx is not None:
                    target[idx] += 1.0
            if target.sum() == 0:
                continue
            target /= 10.0

            # Map VQA-v2 answer_type to a small integer for batched per-type eval.
            atype = ann.get("answer_type", "other")
            atype_idx = ANSWER_TYPE_TO_IDX.get(atype, ANSWER_TYPE_TO_IDX["other"])

            self.samples.append({
                "question": q["question"],
                "image_id": q["image_id"],
                "answer_target": target,
                "answer_type_idx": atype_idx,
            })
            if max_samples is not None and len(self.samples) >= max_samples:
                break

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]

        if self._h5f is None:
            self._h5f = h5py.File(self.h5_path, "r")

        row = self.id_to_row[sample["image_id"]]
        img_array = self._h5f["images"][row]                   # (IMAGE_SIZE, IMAGE_SIZE, 3) uint8
        image = Image.fromarray(img_array)
        image = self.transform(image)

        encoding = self.tokenizer(
            sample["question"],
            max_length=self.max_question_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        input_ids = encoding["input_ids"].squeeze(0)
        attention_mask = encoding["attention_mask"].squeeze(0)

        return (
            image,
            input_ids,
            attention_mask,
            sample["answer_target"],
            torch.tensor(sample["answer_type_idx"], dtype=torch.long),
        )


## Build vocab + datasets + dataloaders


In [ ]:
# Build answer vocab from training annotations
train_ann = DATA_DIR / "answers" / "v2_mscoco_train2014_annotations.json"
answer_to_idx, idx_to_answer = build_answer_vocab(train_ann, NUM_ANSWERS)
print(f"Answer vocab: {len(answer_to_idx)} classes")

train_ds = VQADataset(
    questions_file=DATA_DIR / "questions" / "v2_OpenEnded_mscoco_train2014_questions.json",
    annotations_file=train_ann,
    h5_path=LOCAL_H5_384,
    answer_to_idx=answer_to_idx,
    max_question_len=MAX_QUESTION_LEN,
    transform=get_train_transform(),
    max_samples=MAX_SAMPLES,
)
val_ds = VQADataset(
    questions_file=DATA_DIR / "questions" / "v2_OpenEnded_mscoco_val2014_questions.json",
    annotations_file=DATA_DIR / "answers" / "v2_mscoco_val2014_annotations.json",
    h5_path=LOCAL_H5_384,
    answer_to_idx=answer_to_idx,
    max_question_len=MAX_QUESTION_LEN,
    transform=get_val_transform(),
    max_samples=MAX_SAMPLES,
)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=NUM_WORKERS > 0,
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=NUM_WORKERS > 0,
)

print(f"Train: {len(train_ds):,} samples ({len(train_loader)} batches @ batch={BATCH_SIZE})")
print(f"Val:   {len(val_ds):,} samples ({len(val_loader)} batches)")

with open(CHECKPOINT_DIR / "answer_vocab.json", "w") as f:
    json.dump(answer_to_idx, f)


## CLIP vision encoder (replaces baseline `ImageEncoder`)

Loads `open_clip` ViT-B/16 with LAION-2B weights, **forces image size to 384** (open_clip interpolates the position embedding internally), and exposes the full 577-token sequence (CLS + 576 patches) for cross-attention. The final shared-space `proj` is removed since we have our own projection.

Hard asserts inline catch a silent PE-interpolation failure or width mismatch.


In [ ]:
class CLIPImageEncoder(nn.Module):
    """open_clip ViT-B/16 @ 384, LAION-2B pretrained. Output: (B, 577, embed_dim).

    Manual forward (not visual.output_tokens=True) because that flag drops the
    CLS token; we need CLS + 576 patches for the asymmetric fusion's classifier head.
    """

    def __init__(self, embed_dim=512, freeze=False, image_size=IMAGE_SIZE,
                 model_name=CLIP_MODEL, pretrained=CLIP_PRETRAINED,
                 grad_checkpointing=False):
        super().__init__()
        clip_model, _, _ = open_clip.create_model_and_transforms(
            model_name, pretrained=pretrained, force_image_size=image_size,
        )
        self.visual = clip_model.visual
        # Drop the text tower / logit scale / shared-space projection.
        del clip_model
        self.visual.proj = None

        # Optional activation checkpointing on the transformer (memory <-> compute trade).
        if grad_checkpointing:
            self.visual.set_grad_checkpointing(True)

        if freeze:
            for p in self.visual.parameters():
                p.requires_grad = False

        # visual.width is the transformer hidden size; ViT-B/16 = 768.
        self.projection = nn.Linear(self.visual.width, embed_dim)

        # --- correctness asserts (run once at __init__) ---
        assert self.visual.width == 768, f"expected ViT-B width=768, got {self.visual.width}"
        patch = self.visual.patch_size if isinstance(self.visual.patch_size, int) else self.visual.patch_size[0]
        expected_tokens = (image_size // patch) ** 2 + 1
        actual_tokens = self.visual.positional_embedding.shape[0]
        assert actual_tokens == expected_tokens, (
            f"position embedding interpolation failed: "
            f"got {actual_tokens} tokens, want {expected_tokens} "
            f"(image_size={image_size}, patch={patch})"
        )
        print(f"[CLIPImageEncoder] {model_name} / {pretrained} @ {image_size} -> "
              f"{actual_tokens} tokens, width={self.visual.width}, "
              f"grad_ckpt={grad_checkpointing}")

    def forward(self, images):
        v = self.visual
        x = v.conv1(images)                                    # (B, 768, 24, 24)
        x = x.reshape(x.shape[0], x.shape[1], -1)              # (B, 768, 576)
        x = x.permute(0, 2, 1)                                 # (B, 576, 768)

        # Prepend CLS token. Mirror open_clip's own _expand_token shape pattern
        # (view 1D -> 3D, then expand) so this works across all open_clip versions.
        cls = v.class_embedding.to(x.dtype).view(1, 1, -1).expand(x.shape[0], -1, -1)
        x = torch.cat([cls, x], dim=1)                         # (B, 577, 768)
        x = x + v.positional_embedding.to(x.dtype)
        x = v.patch_dropout(x)
        x = v.ln_pre(x)

        # Modern open_clip (>=2.20) uses batch_first=True; older versions use LND.
        # Detect and transpose if needed so this works on both.
        batch_first = getattr(v.transformer, "batch_first", True)
        if not batch_first:
            x = x.permute(1, 0, 2)                             # NLD -> LND
        x = v.transformer(x)                                   # (B, 577, 768) or (577, B, 768)
        if not batch_first:
            x = x.permute(1, 0, 2)                             # LND -> NLD

        x = v.ln_post(x)
        # v.proj was set to None in __init__; return raw transformer outputs.
        return self.projection(x)                              # (B, 577, embed_dim)


## Text encoder

Identical interface to the baseline `TextEncoder`. The change is **explicit**: we use `last_hidden_state` (the full token sequence) and **do not call `outputs.pooler_output`**. The pooler dense layer is freshly initialized when RoBERTa is loaded (HuggingFace warns about this), so feeding pooler_output into fusion would mix random weights into the gradient path. A one-line diagnostic prints the choice at construction time.


In [ ]:
class TextEncoder(nn.Module):
    """RoBERTa-base text encoder. Returns the full last_hidden_state sequence
    projected to embed_dim — the pooler is bypassed (it would be freshly
    initialized when loading roberta-base via HuggingFace, contributing only
    noise to the gradient).
    """

    ROBERTA_HIDDEN_DIM = 768
    POOLING = "last_hidden_state"  # explicit: full sequence, no pooler.dense

    def __init__(self, embed_dim=512, freeze=False):
        super().__init__()
        self.roberta = RobertaModel.from_pretrained("roberta-base")
        self.projection = nn.Linear(self.ROBERTA_HIDDEN_DIM, embed_dim)

        if freeze:
            for p in self.roberta.parameters():
                p.requires_grad = False

        # Diagnostic: makes the pooler-bypass explicit in the run log.
        assert hasattr(self.roberta, "pooler"), \
            "expected RoBERTa to expose a pooler (which we intentionally bypass)"
        print(f"[TextEncoder] pooling={self.POOLING} (pooler.dense bypassed). "
              f"hidden_dim={self.roberta.config.hidden_size}")

    def forward(self, input_ids, attention_mask):
        out = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        # Use full sequence; do NOT touch out.pooler_output.
        return self.projection(out.last_hidden_state)          # (B, seq_len, embed_dim)


## Cross-attention blocks + asymmetric fusion + classifier head

These three classes are **verbatim** from the baseline notebook (cells 28 and 30) so the research contribution is unchanged. Only the encoder feeding the fusion changes.


In [ ]:
class CrossAttentionBlock(nn.Module):
    """Cross-attention: queries from one modality attend to keys/values from another.
    Pre-norm transformer block with attention + FFN, both with residuals.
    """

    def __init__(self, embed_dim, num_heads=8, dropout=0.1, attn_dropout=None):
        super().__init__()
        if attn_dropout is None:
            attn_dropout = dropout
        self.norm_q = nn.LayerNorm(embed_dim)
        self.norm_kv = nn.LayerNorm(embed_dim)
        self.cross_attn = nn.MultiheadAttention(
            embed_dim, num_heads, dropout=attn_dropout, batch_first=True)
        self.norm_ff = nn.LayerNorm(embed_dim)
        self.ff = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * 4),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(embed_dim * 4, embed_dim),
            nn.Dropout(dropout),
        )

    def forward(self, query, key_value, key_padding_mask=None):
        q = self.norm_q(query)
        kv = self.norm_kv(key_value)
        attended, attn_weights = self.cross_attn(
            q, kv, kv, key_padding_mask=key_padding_mask,
            need_weights=True, average_attn_weights=True)
        query = query + attended
        query = query + self.ff(self.norm_ff(query))
        return query, attn_weights


class AsymmetricCrossModalFusion(nn.Module):
    """Two SEPARATE cross-attention blocks with DIFFERENT learned weights,
    executed SEQUENTIALLY -- core research contribution from the baseline.

    Step 1: image attends to text  -> img_attended  (visually-grounded image features)
    Step 2: text attends to img_attended (NOT raw image) -> txt_attended
    """

    def __init__(self, embed_dim, num_heads=8, dropout=0.1, attn_dropout=None):
        super().__init__()
        self.cross_attn_img_to_txt = CrossAttentionBlock(embed_dim, num_heads, dropout, attn_dropout)
        self.cross_attn_txt_to_img = CrossAttentionBlock(embed_dim, num_heads, dropout, attn_dropout)

    def forward(self, image_features, text_features, text_padding_mask=None):
        img_attended, attn_i2t = self.cross_attn_img_to_txt(
            query=image_features, key_value=text_features,
            key_padding_mask=text_padding_mask)
        txt_attended, attn_t2i = self.cross_attn_txt_to_img(
            query=text_features, key_value=img_attended)
        return img_attended, txt_attended, attn_i2t, attn_t2i


class AsymmetricVQAModelE2E(nn.Module):
    """End-to-end: CLIP vision encoder + RoBERTa text encoder + asymmetric fusion + classifier."""

    def __init__(self, num_answers, embed_dim=512, num_heads=8, dropout=0.3,
                 attn_dropout=None, cls_dropout=None,
                 image_size=IMAGE_SIZE,
                 clip_model_name=CLIP_MODEL, clip_pretrained=CLIP_PRETRAINED,
                 vision_grad_checkpointing=False):
        super().__init__()
        if cls_dropout is None:
            cls_dropout = dropout
        self.image_encoder = CLIPImageEncoder(
            embed_dim=embed_dim, freeze=False,
            image_size=image_size, model_name=clip_model_name,
            pretrained=clip_pretrained,
            grad_checkpointing=vision_grad_checkpointing,
        )
        self.text_encoder = TextEncoder(embed_dim=embed_dim, freeze=False)
        self.fusion = AsymmetricCrossModalFusion(embed_dim, num_heads, dropout, attn_dropout)
        self.classifier = nn.Sequential(
            nn.Linear(embed_dim * 2, embed_dim), nn.ReLU(),
            nn.Dropout(cls_dropout), nn.Linear(embed_dim, num_answers))

    def forward(self, images, input_ids, attention_mask):
        img = self.image_encoder(images)                       # (B, 577, embed_dim)
        txt = self.text_encoder(input_ids, attention_mask)     # (B, seq_len, embed_dim)
        text_pad_mask = attention_mask == 0
        img_att, txt_att, attn_i2t, attn_t2i = self.fusion(img, txt, text_pad_mask)
        z = torch.cat([img_att[:, 0, :], txt_att[:, 0, :]], dim=-1)  # CLS-of-each, concatenated
        logits = self.classifier(z)
        return logits, {"img_to_txt": attn_i2t, "txt_to_img": attn_t2i}


## EMA wrapper

Tracks an exponential moving average of all floating-point parameters (and buffers — including BatchNorm running stats / RoBERTa LayerNorm stats — to keep eval consistent). Decay 0.9998 ≈ ~5000-step half-life on a 0.0002 update rate. Eval uses EMA weights; saves both `*_live.pt` and `*_ema.pt`.


In [ ]:
class ModelEMA:
    """Exponential moving average over a model's float tensors (params + buffers)."""

    def __init__(self, model: nn.Module, decay: float = 0.9998):
        self.decay = decay
        self.steps = 0
        # Clone every float tensor in the state_dict. Integer buffers (e.g. token type ids)
        # are skipped — they don't move.
        self.shadow: dict[str, torch.Tensor] = {
            n: p.detach().clone()
            for n, p in model.state_dict().items()
            if torch.is_floating_point(p)
        }

    @torch.no_grad()
    def update(self, model: nn.Module):
        d = self.decay
        msd = model.state_dict()
        any_moved = False
        for n, shadow_p in self.shadow.items():
            live = msd[n].detach()
            # In-place EMA: shadow = decay * shadow + (1-decay) * live
            if shadow_p.dtype != live.dtype:
                live = live.to(shadow_p.dtype)
            before = shadow_p.clone() if self.steps == 0 else None  # for first-step assert
            shadow_p.mul_(d).add_(live, alpha=1.0 - d)
            if before is not None and not torch.equal(shadow_p, before):
                any_moved = True
        if self.steps == 0:
            assert any_moved, "EMA shadow did not move on first update — check that param dtypes are float."
        self.steps += 1

    @torch.no_grad()
    def copy_to(self, model: nn.Module):
        """Overwrite model's float tensors with EMA values (in-place)."""
        msd = model.state_dict()
        for n, shadow_p in self.shadow.items():
            msd[n].copy_(shadow_p)

    def state_dict(self):
        return {"decay": self.decay, "steps": self.steps, "shadow": self.shadow}

    def load_state_dict(self, sd):
        self.decay = sd["decay"]
        self.steps = sd["steps"]
        self.shadow = sd["shadow"]


class _SwapEMA:
    """Context manager that swaps live weights for EMA, runs eval, then restores live."""

    def __init__(self, model: nn.Module, ema: "ModelEMA"):
        self.model = model
        self.ema = ema
        self._backup: dict[str, torch.Tensor] = {}

    def __enter__(self):
        # Snapshot live float tensors, then copy EMA in.
        msd = self.model.state_dict()
        for n in self.ema.shadow:
            self._backup[n] = msd[n].detach().clone()
        self.ema.copy_to(self.model)
        return self.model

    def __exit__(self, exc_type, exc, tb):
        msd = self.model.state_dict()
        for n, v in self._backup.items():
            msd[n].copy_(v)
        self._backup.clear()


## Sanity check — model build, shapes, params

Builds the model on CPU first (cheap), runs a dummy forward, verifies the vision and text sequence shapes, then verifies that EMA actually moves on a fake step. If any of these fail, training will not work — better to catch it here in 5 seconds than 5 minutes in.


In [ ]:
# Sanity-check on CPU before allocating GPU memory.
_dummy_model = AsymmetricVQAModelE2E(
    num_answers=NUM_ANSWERS, embed_dim=EMBED_DIM, num_heads=NUM_HEADS, dropout=DROPOUT,
    attn_dropout=ATTN_DROPOUT, cls_dropout=CLS_DROPOUT,
    image_size=IMAGE_SIZE, vision_grad_checkpointing=False,
)
_dummy_model.eval()

with torch.no_grad():
    _img = torch.randn(2, 3, IMAGE_SIZE, IMAGE_SIZE)
    _ids = torch.zeros(2, MAX_QUESTION_LEN, dtype=torch.long)
    _mask = torch.ones(2, MAX_QUESTION_LEN, dtype=torch.long)
    _logits, _attn = _dummy_model(_img, _ids, _mask)

    _vision_out = _dummy_model.image_encoder(_img)
    _text_out = _dummy_model.text_encoder(_ids, _mask)

assert _vision_out.shape == (2, 577, EMBED_DIM), f"vision out: {_vision_out.shape}"
assert _text_out.shape   == (2, MAX_QUESTION_LEN, EMBED_DIM), f"text out: {_text_out.shape}"
assert _logits.shape     == (2, NUM_ANSWERS), f"logits: {_logits.shape}"

n_train = sum(p.numel() for p in _dummy_model.parameters() if p.requires_grad)
n_total = sum(p.numel() for p in _dummy_model.parameters())
print(f"[Sanity] vision={tuple(_vision_out.shape)} text={tuple(_text_out.shape)} "
      f"logits={tuple(_logits.shape)}")
print(f"[Sanity] trainable params: {n_train:,} / {n_total:,} total")

# EMA smoke test: one update on dummy weights should move the shadow.
_ema = ModelEMA(_dummy_model, decay=EMA_DECAY)
with torch.no_grad():
    # Perturb a couple of weights so live != initial shadow
    for p in _dummy_model.parameters():
        p.add_(torch.randn_like(p) * 1e-3)
        break
_ema.update(_dummy_model)
print(f"[Sanity] EMA steps={_ema.steps} (after one fake step)")

del _dummy_model, _ema, _img, _ids, _mask, _logits, _attn, _vision_out, _text_out
import gc; gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


## Train / eval helpers

- **`train_one_epoch_amp`**: modern `torch.amp` API, gradient accumulation, EMA update on each optimizer step, scheduler steps on optimizer step, peak-memory print after the first optimizer step (verifies headroom before committing to the run).
- **`evaluate_with_types`**: returns `val_loss`, `val_vqa_acc` (overall), and `val_vqa_acc_{yesno,number,other}`. Optionally swaps in EMA weights.


In [ ]:
def _amp_ctx(use_amp: bool):
    return autocast("cuda") if use_amp else nullcontext()


def train_one_epoch_amp(model, loader, criterion, optimizer, scaler, use_amp,
                        scheduler=None, ema: "ModelEMA | None" = None,
                        accum_steps: int = 1, grad_clip_norm: float | None = None,
                        log_peak_mem: bool = False):
    model.train()
    optimizer.zero_grad(set_to_none=True)

    total_loss = 0.0
    correct = 0
    total = 0
    micro_in_batch = 0
    peak_mem_logged = not log_peak_mem  # if False we still need to log; True means we already did

    pbar = tqdm(loader, desc="  train", leave=False)
    for step, batch in enumerate(pbar):
        images, ids, masks, answers, _atype = batch
        images  = images.to(device, non_blocking=True)
        ids     = ids.to(device, non_blocking=True)
        masks   = masks.to(device, non_blocking=True)
        answers = answers.to(device, non_blocking=True)

        with _amp_ctx(use_amp):
            logits, _ = model(images, ids, masks)
            loss = criterion(logits, answers)
            loss_scaled = loss / accum_steps  # average across the accumulation window

        if scaler is not None:
            scaler.scale(loss_scaled).backward()
        else:
            loss_scaled.backward()

        micro_in_batch += 1
        is_step = (micro_in_batch == accum_steps) or (step == len(loader) - 1)

        if is_step:
            if scaler is not None:
                if grad_clip_norm is not None:
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip_norm)
                scaler.step(optimizer)
                scaler.update()
            else:
                if grad_clip_norm is not None:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip_norm)
                optimizer.step()
            optimizer.zero_grad(set_to_none=True)

            if scheduler is not None:
                scheduler.step()
            if ema is not None:
                ema.update(model)

            if not peak_mem_logged and torch.cuda.is_available():
                peak_gb = torch.cuda.max_memory_allocated() / 1e9
                print(f"[Mem] peak after first optimizer step: {peak_gb:.2f} GB "
                      f"(batch={BATCH_SIZE}, accum={accum_steps}, ckpt={USE_GRAD_CHECKPOINT})")
                peak_mem_logged = True

            micro_in_batch = 0

        # Bookkeeping uses the un-scaled loss for human readability.
        total_loss += loss.item() * answers.size(0)
        correct += (logits.argmax(dim=1) == answers.argmax(dim=1)).sum().item()
        total += answers.size(0)

    return {"train_loss": total_loss / total, "train_acc": correct / total * 100}


@torch.no_grad()
def evaluate_with_types(model, loader, criterion, use_amp,
                        ema: "ModelEMA | None" = None):
    """Eval that returns overall VQA acc + per-type (yes/no, number, other).
    If `ema` is provided, evaluation runs with EMA weights swapped in (then restored).
    """
    swap = _SwapEMA(model, ema) if ema is not None else nullcontext()
    with swap:
        model.eval()
        total_loss = 0.0
        vqa_acc_sum = 0.0
        total = 0

        per_type_sum = np.zeros(len(ANSWER_TYPES), dtype=np.float64)
        per_type_cnt = np.zeros(len(ANSWER_TYPES), dtype=np.int64)

        for images, ids, masks, answers, atypes in tqdm(loader, desc="  eval", leave=False):
            images  = images.to(device, non_blocking=True)
            ids     = ids.to(device, non_blocking=True)
            masks   = masks.to(device, non_blocking=True)
            answers = answers.to(device, non_blocking=True)
            atypes_np = atypes.numpy()

            with _amp_ctx(use_amp):
                logits, _ = model(images, ids, masks)
                loss = criterion(logits, answers)

            total_loss += loss.item() * answers.size(0)

            preds = logits.argmax(dim=1)
            pred_soft = answers[torch.arange(answers.size(0), device=answers.device), preds]
            vqa_scores = torch.clamp(pred_soft * 10.0 / 3.0, max=1.0).detach().cpu().numpy()

            vqa_acc_sum += float(vqa_scores.sum())
            total += answers.size(0)

            # Aggregate per-type
            for t_idx in range(len(ANSWER_TYPES)):
                mask = atypes_np == t_idx
                if mask.any():
                    per_type_sum[t_idx] += float(vqa_scores[mask].sum())
                    per_type_cnt[t_idx] += int(mask.sum())

    out = {
        "val_loss": total_loss / max(1, total),
        "val_vqa_acc": vqa_acc_sum / max(1, total) * 100,
    }
    for t_idx, t_name in enumerate(ANSWER_TYPES):
        suffix = t_name.replace("/", "")  # "yes/no" -> "yesno"
        out[f"val_vqa_acc_{suffix}"] = (
            per_type_sum[t_idx] / per_type_cnt[t_idx] * 100 if per_type_cnt[t_idx] else float("nan")
        )
    return out


## `run_training`

Same param-group split + AdamW + cosine-with-warmup as baseline, but:
- Warmup is **5% of total optimizer steps** (not 1 epoch).
- EMA tracked from step 0; best checkpoint chosen by EMA val acc.
- Per-type metrics printed and logged each epoch.
- Saves `*_live.pt` (live weights, for resume) and `*_ema.pt` (EMA, for eval).


In [ ]:
def run_training(model_type: str = "asymmetric", run_name: str | None = None,
                 resume_from: str | Path | None = None):
    set_seed(SEED)
    if run_name is None:
        run_name = f"{model_type}_clip384_s{SEED}"

    # --- Build model ---
    assert model_type == "asymmetric", "this notebook only trains the asymmetric variant"
    model = AsymmetricVQAModelE2E(
        num_answers=NUM_ANSWERS, embed_dim=EMBED_DIM, num_heads=NUM_HEADS, dropout=DROPOUT,
        attn_dropout=ATTN_DROPOUT, cls_dropout=CLS_DROPOUT,
        image_size=IMAGE_SIZE,
        clip_model_name=CLIP_MODEL, clip_pretrained=CLIP_PRETRAINED,
        vision_grad_checkpointing=USE_GRAD_CHECKPOINT,
    ).to(device)

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())
    print(f"\nModel: {model_type} | Trainable params: {trainable:,} / {total_params:,} total")

    # --- Optimizer + scheduler ---
    criterion = nn.BCEWithLogitsLoss()

    NO_DECAY_KEYS = ("bias", "LayerNorm.weight", "layer_norm.weight", "ln_")
    def _split(named_params):
        decay, nodecay = [], []
        for n, p in named_params:
            (nodecay if any(k in n for k in NO_DECAY_KEYS) else decay).append(p)
        return decay, nodecay

    enc_named   = [(n, p) for n, p in model.named_parameters()
                   if p.requires_grad and ("image_encoder" in n or "text_encoder" in n)]
    other_named = [(n, p) for n, p in model.named_parameters()
                   if p.requires_grad and not ("image_encoder" in n or "text_encoder" in n)]
    enc_decay,   enc_nodecay   = _split(enc_named)
    other_decay, other_nodecay = _split(other_named)

    print(f"  Encoder params:    {sum(p.numel() for p in enc_decay) + sum(p.numel() for p in enc_nodecay):,} (lr={ENCODER_LR})")
    print(f"  Fusion+cls params: {sum(p.numel() for p in other_decay) + sum(p.numel() for p in other_nodecay):,} (lr={LEARNING_RATE})")

    optimizer = torch.optim.AdamW([
        {"params": enc_decay,     "lr": ENCODER_LR,    "weight_decay": WEIGHT_DECAY},
        {"params": enc_nodecay,   "lr": ENCODER_LR,    "weight_decay": 0.0},
        {"params": other_decay,   "lr": LEARNING_RATE, "weight_decay": WEIGHT_DECAY},
        {"params": other_nodecay, "lr": LEARNING_RATE, "weight_decay": 0.0},
    ])

    # Optimizer steps per epoch = ceil(len(train_loader) / accum)
    steps_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM_STEPS)
    total_steps  = steps_per_epoch * EPOCHS
    warmup_steps = max(1, int(round(total_steps * WARMUP_FRAC)))
    print(f"  Optimizer steps/epoch: {steps_per_epoch} | total: {total_steps} | warmup: {warmup_steps} ({WARMUP_FRAC*100:.1f}%)")

    def lr_lambda(step):
        if step < warmup_steps:
            return step / max(1, warmup_steps)
        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return 0.5 * (1.0 + math.cos(math.pi * progress))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

    use_amp = USE_AMP and device.type == "cuda"
    scaler = GradScaler("cuda") if use_amp else None

    ema = ModelEMA(model, decay=EMA_DECAY) if USE_EMA else None

    history = []
    best_val_acc = 0.0
    best_acc_epoch = None
    start_epoch = 1

    # --- Resume ---
    if resume_from is not None:
        checkpoint_path = Path(resume_from)
        if checkpoint_path.exists():
            print(f"Resuming from {checkpoint_path}")
            ckpt = torch.load(checkpoint_path, map_location=device)
            model.load_state_dict(ckpt["model_state_dict"])
            optimizer.load_state_dict(ckpt["optimizer_state_dict"])
            if scheduler is not None and ckpt.get("scheduler_state_dict") is not None:
                scheduler.load_state_dict(ckpt["scheduler_state_dict"])
            if ema is not None and ckpt.get("ema_state_dict") is not None:
                ema.load_state_dict(ckpt["ema_state_dict"])
            start_epoch = ckpt["epoch"] + 1

            history_file = METRICS_DIR / f"{run_name}_history.json"
            if history_file.exists():
                with open(history_file, "r") as f:
                    history = json.load(f)
            if history:
                best_entry = max(history, key=lambda h: h.get("val_vqa_acc", 0))
                best_val_acc = best_entry.get("val_vqa_acc", 0)
                best_acc_epoch = best_entry["epoch"]
        else:
            print(f"Checkpoint not found at {checkpoint_path}. Training from scratch.")

    # --- Training loop ---
    for epoch in range(start_epoch, EPOCHS + 1):
        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()

        t0 = time.time()
        train_m = train_one_epoch_amp(
            model, train_loader, criterion, optimizer, scaler, use_amp,
            scheduler=scheduler, ema=ema,
            accum_steps=GRAD_ACCUM_STEPS, grad_clip_norm=GRAD_CLIP_NORM,
            log_peak_mem=(epoch == start_epoch),
        )
        # Eval on EMA weights (best-effort accuracy estimate of the EMA snapshot)
        val_m_ema = evaluate_with_types(model, val_loader, criterion, use_amp, ema=ema)
        # Also eval on live weights for comparison
        val_m_live = evaluate_with_types(model, val_loader, criterion, use_amp, ema=None)

        elapsed = time.time() - t0

        epoch_data = {
            "epoch": epoch, **train_m,
            "val_loss":            val_m_ema["val_loss"],
            "val_vqa_acc":         val_m_ema["val_vqa_acc"],          # EMA -> best-checkpoint selection metric
            "val_vqa_acc_yesno":   val_m_ema["val_vqa_acc_yesno"],
            "val_vqa_acc_number":  val_m_ema["val_vqa_acc_number"],
            "val_vqa_acc_other":   val_m_ema["val_vqa_acc_other"],
            "val_vqa_acc_live":    val_m_live["val_vqa_acc"],
            "elapsed_s": round(elapsed, 1),
        }
        history.append(epoch_data)

        print(
            f"  Epoch {epoch}/{EPOCHS} | loss {train_m['train_loss']:.4f} | "
            f"train {train_m['train_acc']:.2f}% | "
            f"vqa_acc(ema) {val_m_ema['val_vqa_acc']:.2f}% "
            f"(live {val_m_live['val_vqa_acc']:.2f}%) | "
            f"yesno {val_m_ema['val_vqa_acc_yesno']:.2f}% | "
            f"number {val_m_ema['val_vqa_acc_number']:.2f}% | "
            f"other {val_m_ema['val_vqa_acc_other']:.2f}% | "
            f"{elapsed:.0f}s"
        )

        # Save full live checkpoint (for resume) — every epoch, only latest+best kept
        live_ckpt = CHECKPOINT_DIR / f"{run_name}_epoch{epoch}.pt"
        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict() if scheduler is not None else None,
            "ema_state_dict": ema.state_dict() if ema is not None else None,
            "metrics": epoch_data,
        }, live_ckpt)

        if val_m_ema["val_vqa_acc"] > best_val_acc:
            best_val_acc = val_m_ema["val_vqa_acc"]
            best_acc_epoch = epoch
            # Live best (weights only)
            torch.save({"model_state_dict": model.state_dict(), **epoch_data},
                       CHECKPOINT_DIR / f"{run_name}_best_live.pt")
            # EMA best (weights only) — load EMA into the model, snapshot, then restore
            with _SwapEMA(model, ema) if ema is not None else nullcontext():
                torch.save({"model_state_dict": model.state_dict(), **epoch_data},
                           CHECKPOINT_DIR / f"{run_name}_best_ema.pt")
            print(f"    New best (EMA): {best_val_acc:.2f}%")

        # Cap disk: keep latest + best
        keep = {live_ckpt}
        if best_acc_epoch is not None:
            keep.add(CHECKPOINT_DIR / f"{run_name}_epoch{best_acc_epoch}.pt")
        for stale in CHECKPOINT_DIR.glob(f"{run_name}_epoch*.pt"):
            if stale not in keep:
                stale.unlink(missing_ok=True)

        with open(METRICS_DIR / f"{run_name}_history.json", "w") as f:
            json.dump(history, f, indent=2)

    print(f"\nTraining complete. Best EMA val_vqa_acc: {best_val_acc:.2f}%")
    return model, history, ema


### Launch training

Auto-resumes from the highest-numbered `*_epoch{N}.pt` if one exists.


In [ ]:
# Auto-resume: pick the highest-numbered _clip384_s42_epoch*.pt if any.
ckpts = sorted(
    CHECKPOINT_DIR.glob("asymmetric_clip384_s42_epoch*.pt"),
    key=lambda p: int(p.stem.rsplit("epoch", 1)[1]),
)
resume_path = ckpts[-1] if ckpts else None
print(f"Resume from: {resume_path}")

asymmetric_model, asymmetric_history, asymmetric_ema = run_training(
    "asymmetric", resume_from=resume_path,
)


In [ ]:
# Optional: persist results to Drive after the run.
# !mkdir -p /content/drive/MyDrive/clip_upgrade_results/
# !cp -r /content/results/ /content/drive/MyDrive/clip_upgrade_results/


### Evaluate the best EMA checkpoint

Load `*_best_ema.pt` and report overall + per-type val accuracy. This is the canonical "did we beat 64.27?" number — use it to compare against the baseline.


In [ ]:
# Rebuild a fresh model and load the best EMA weights.
best_ema_path = CHECKPOINT_DIR / "asymmetric_clip384_s42_best_ema.pt"
assert best_ema_path.exists(), f"missing {best_ema_path}"

eval_model = AsymmetricVQAModelE2E(
    num_answers=NUM_ANSWERS, embed_dim=EMBED_DIM, num_heads=NUM_HEADS, dropout=DROPOUT,
    attn_dropout=ATTN_DROPOUT, cls_dropout=CLS_DROPOUT,
    image_size=IMAGE_SIZE,
    clip_model_name=CLIP_MODEL, clip_pretrained=CLIP_PRETRAINED,
    vision_grad_checkpointing=False,
).to(device)
state = torch.load(best_ema_path, map_location=device)
eval_model.load_state_dict(state["model_state_dict"])
eval_model.eval()

criterion = nn.BCEWithLogitsLoss()
use_amp = USE_AMP and device.type == "cuda"
metrics = evaluate_with_types(eval_model, val_loader, criterion, use_amp)

# Pretty print
rows = [
    ("Overall val_vqa_acc", metrics["val_vqa_acc"]),
    ("  Yes/No",            metrics["val_vqa_acc_yesno"]),
    ("  Number",            metrics["val_vqa_acc_number"]),
    ("  Other",             metrics["val_vqa_acc_other"]),
    ("val_loss",            metrics["val_loss"]),
]
print(f"\nBest EMA checkpoint: {best_ema_path.name}")
for k, v in rows:
    if k == "val_loss":
        print(f"{k:<24} {v:.4f}")
    else:
        print(f"{k:<24} {v:.2f}%")


### Plots: loss / overall vs per-type accuracy curves


In [ ]:
histories = {"Asymmetric_CLIP384": asymmetric_history}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for name, hist in histories.items():
    epochs = [h["epoch"] for h in hist]
    # Loss
    axes[0].plot(epochs, [h["train_loss"] for h in hist], label=f"{name} train")
    axes[0].plot(epochs, [h["val_loss"]   for h in hist], "--", label=f"{name} val")
    # Accuracy (overall + per-type)
    axes[1].plot(epochs, [h["train_acc"]            for h in hist],        label=f"{name} train acc")
    axes[1].plot(epochs, [h["val_vqa_acc"]          for h in hist], "--",  label=f"{name} val (overall, EMA)")
    axes[1].plot(epochs, [h["val_vqa_acc_yesno"]    for h in hist], ":",   label=f"{name} val yes/no")
    axes[1].plot(epochs, [h["val_vqa_acc_number"]   for h in hist], ":",   label=f"{name} val number")
    axes[1].plot(epochs, [h["val_vqa_acc_other"]    for h in hist], ":",   label=f"{name} val other")

axes[0].set(xlabel="Epoch", ylabel="Loss", title="Loss")
axes[1].set(xlabel="Epoch", ylabel="Accuracy (%)", title="VQA Accuracy (overall + per-type)")
for ax in axes:
    ax.legend(fontsize=8, loc="best")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "training_curves_clip384.png", dpi=150, bbox_inches="tight")
plt.show()


### Final per-type bar chart

End-of-training per-type accuracy from the best EMA checkpoint. Use this with the baseline notebook's equivalent chart to localize where the CLIP-upgrade win is coming from.


In [ ]:
cats = ["yes/no", "number", "other"]
vals = [metrics[f"val_vqa_acc_{c.replace('/', '')}"] for c in cats]

fig, ax = plt.subplots(figsize=(7, 5))
bars = ax.bar(cats, vals, color="steelblue")
for bar in bars:
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.4,
            f"{bar.get_height():.2f}", ha="center", va="bottom", fontsize=10)
ax.set_ylabel("VQA Accuracy (%)")
ax.set_title("Per-question-type val accuracy (CLIP@384, best EMA)")
ax.axhline(metrics["val_vqa_acc"], color="gray", linestyle="--", alpha=0.7,
           label=f"Overall: {metrics['val_vqa_acc']:.2f}%")
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES_DIR / "per_type_clip384.png", dpi=150, bbox_inches="tight")
plt.show()


## Test Set Predictions Export

Run inference on the VQA v2.0 **test2015** split (no annotations) and write predictions to JSON in the official VQA submission format: `[{"question_id": int, "answer": str}, ...]`.

Test images aren't in the 384px H5 cache (`vqa_images_384.h5` only covers train+val), so we load raw JPEGs from `images/test2015/`, resize+center-crop to `IMAGE_SIZE`, and CLIP-normalize on-the-fly. Inference uses `eval_model`, already loaded with the best EMA weights from the cell above.

In [ ]:
class VQATestDataset(Dataset):
    """VQA test split: questions + raw JPEGs from images/test2015/, no annotations.

    Test images are not in the 384px H5 cache, so we resize+center-crop on-the-fly
    and apply the same CLIP normalization used at validation.
    """

    def __init__(self, questions_file, images_dir, max_question_len=MAX_QUESTION_LEN,
                 image_size=IMAGE_SIZE, max_samples=None):
        self.images_dir = Path(images_dir) / "test2015"
        self.max_question_len = max_question_len
        self.transform = transforms.Compose([
            transforms.Resize(image_size, interpolation=transforms.InterpolationMode.BICUBIC),
            transforms.CenterCrop(image_size),
            transforms.ToTensor(),
            transforms.Normalize(mean=CLIP_MEAN, std=CLIP_STD),
        ])
        self.tokenizer = RobertaTokenizer.from_pretrained("roberta-base")

        with open(questions_file) as f:
            self.samples = json.load(f)["questions"]
        if max_samples is not None:
            self.samples = self.samples[:max_samples]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        # test-dev2015 reuses the test2015 image pool, so the filename prefix is
        # always COCO_test2015_* regardless of which question split we loaded.
        img_path = self.images_dir / f"COCO_test2015_{sample['image_id']:012d}.jpg"
        image = Image.open(img_path).convert("RGB")
        image = self.transform(image)

        encoding = self.tokenizer(
            sample["question"],
            max_length=self.max_question_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        return (
            image,
            encoding["input_ids"].squeeze(0),
            encoding["attention_mask"].squeeze(0),
            sample["question_id"],
        )


TEST_QUESTIONS_FILE = DATA_DIR / "questions" / "v2_OpenEnded_mscoco_test2015_questions.json"

test_ds = VQATestDataset(
    questions_file=TEST_QUESTIONS_FILE,
    images_dir=DATA_DIR / "images",
    max_question_len=MAX_QUESTION_LEN,
    image_size=IMAGE_SIZE,
    max_samples=None,  # always export the full test split, regardless of dev MAX_SAMPLES
)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=True)
print(f"Test: {len(test_ds):,} samples ({len(test_loader)} batches)")

In [ ]:
@torch.no_grad()
def predict_test(model, loader, idx_to_answer):
    """Return [{question_id, answer}, ...] in official VQA submission format."""
    model.eval()
    use_amp = USE_AMP and device.type == "cuda"
    predictions = []

    for images, input_ids, attention_mask, question_ids in tqdm(loader, desc="predict"):
        images         = images.to(device, non_blocking=True)
        input_ids      = input_ids.to(device, non_blocking=True)
        attention_mask = attention_mask.to(device, non_blocking=True)

        with _amp_ctx(use_amp):
            logits, _ = model(images, input_ids, attention_mask)

        preds = logits.argmax(dim=1).cpu().tolist()
        for qid, p in zip(question_ids.tolist(), preds):
            predictions.append({"question_id": int(qid), "answer": idx_to_answer[int(p)]})

    return predictions


PREDICTIONS_DIR = CHECKPOINT_DIR.parent / "predictions"
PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)

preds = predict_test(eval_model, test_loader, idx_to_answer)
out_path = PREDICTIONS_DIR / "asymmetric_clip384_test_predictions.json"
with open(out_path, "w") as f:
    json.dump(preds, f)
print(f"Saved {len(preds):,} predictions -> {out_path}")